In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
#from langchain_community.chat_models import ChatOllama
from langchain_ollama import ChatOllama
from mcp_astro_chatbot import MCP_ChatBot
import json
import asyncio

# --- LLM ---
llm = ChatOllama(model="llama3")



In [ ]:
# --- State ---
class AgentState(TypedDict):
    messages: list
    stage: str
    intent: str
    profile_matching_inprogress: bool
    login_successful: bool
    boy_profile_fetch_success: bool
    girl_profile_fetch_success: bool
    boy_profile_url: str
    girl_profile_url: str
    boy_profile: dict
    girl_profile: dict
    mcp_client: MCP_ChatBot



# --- Nodes ---

def intent_detection(state: AgentState):
    text = state["message"].lower()

    if "match" in text:
        intent = "matchmaking"
    else:
        intent = "general"

    return {**state, "intent": intent}


def general_chat(state: AgentState):
    prompt = f"User: {state['message']}\nAssistant:"
    res = llm.invoke(prompt)

    return {**state, "response": res.content}


def matchmaking_node(state: AgentState):
    prompt = f"""
    You are a matrimonial assistant. Analyse the given Boy and Girl profiles for matrimonial compatibility using instructions:

    1. Astrological inputs or Kundali matching inputs are available in Boy's profile data. Consider them as important criteria.
    2. Do not use any Astrological inputs/match making inputs from girl's profile, as boy's profile carries all the details.
    3. Use About Me and About Family fields from both profiles to check comptability
    4. Use Hobbies to check compatibility
    5. Use multiple expactation fields and cross check between each others profiles. e.g. Expectations from Boy's profile should be matched against parameters from Girl's profile and vice versa.
       Example: Expected marital status from Boy's profile should be checked with Marital Status of girl.
    6. Check family finanacial status compatibility
    7. Check comptability between Age and Heights
    
    Use the JSON profiles attached.

    Structure your output as markdown with different heading for each points #1 to #6.

    Boy profile: {state["boy_profile"]}
    Girl profile: {state["girl_profile"]}
    """

    res = llm.invoke(prompt)

    return {**state, "response": res.content}


async def profile_site_login(state: AgentState):
    print("Entered Profile Site Login Node")
    login_result = await state["mcp_client"].excute_profile_login()
    if len(login_result) > 0 and "success" in login_result[0]["text"].lower():
        print("Profile Site Login successful")
        state["login_successful"] = True
    else:
        print("Profile Login not successful")       
        state["login_successful"] = False
    
    return {**state} 


async def boy_profile_fetch(state: AgentState):
    print("Entered boy_profile_fetch Node")
    login_result = await state["mcp_client"].excute_profile_fetch_tool(state["boy_profile_url"], False, True, False)
    if len(login_result) > 0 and len(login_result[0]["text"].lower())>0:
        profile_data = []
        try:
            for ext_rslt in login_result:
                profile = json.loads(ext_rslt["text"])
                if len(profile.keys()) > 0:       
                    profile_data.append(profile)
            print("boy_profile_fetch successful: ")
            state["boy_profile"] = profile_data[0]
            state["boy_profile_fetch_success"] = True
        except Exception as e:
            print("Exception happened in boy_profile_fetch", e)
            state["boy_profile_fetch_success"] = False

    else:
        print("Profile Login not successful")       
        state["boy_profile_fetch_success"] = False
    
    return {**state}  

async def girl_profile_fetch(state: AgentState):
    print("Entered girl_profile_fetch Node")
    login_result = await state["mcp_client"].excute_profile_fetch_tool(state["girl_profile_url"], False, False, True)
    if len(login_result) > 0 and len(login_result[0]["text"].lower())>0:
        profile_data = []
        try:
            for ext_rslt in login_result:
                profile = json.loads(ext_rslt["text"])
                if len(profile.keys()) > 0:       
                    profile_data.append(profile)
            print("girl_profile_fetch successful: ")
            state["girl_profile"] = profile_data[0]
            state["girl_profile_fetch_success"] = True
        except Exception as e:
            print("Exception happened in girl_profile_fetch", e)
            state["girl_profile_fetch_success"] = False

    else:
        print("Girl Profile fetch not successful")       
        state["girl_profile_fetch_success"] = False
    
    return {**state}         


# --- Router ---
def route(state: AgentState):
    return state["intent"]

def check_login_success(state: AgentState):
    if state["login_successful"]:
        return "login_successful"
    else:
        return "login_failed"
    
def check_boy_profile_fetch_success(state: AgentState):
    if state["boy_profile_fetch_success"]:
        return "successful"
    else:
        return "failed"


def check_girl_profile_fetch_success(state: AgentState):
    if state["girl_profile_fetch_success"]:
        return "successful"
    else:
        return "failed"

async def agent_main():
    mcp_client = MCP_ChatBot(15)
    session_state = AgentState()
    session_state["mcp_client"] = mcp_client

    # Trial
    session_state["profile_matching_inprogress"] = True
    session_state["boy_profile_url"] = "https://www.anuroopwiwaha.com/User/MemberProfile.aspx?member_id=870081"
    session_state["girl_profile_url"] = "https://www.anuroopwiwaha.com/user/view_profile.aspx"
    session_state["message"] = "Match these two profiles based on values and career goals"
    try:
        await mcp_client.connect_to_remote_server()
        graph = construct_graph()

        result = await graph.ainvoke(session_state)

        print("Whole State", result)
        print(result["response"])
        
    except Exception as e:
        print("Agent Main encountered exception",e)
    finally:
        await mcp_client.cleanup()

def construct_graph():
    # --- Graph ---
    builder = StateGraph(AgentState)
    builder.add_node("intent_detection", intent_detection)
    builder.add_node("general", general_chat)
    builder.add_node("profile_site_login", profile_site_login)
    builder.add_node("boy_profile_fetch", boy_profile_fetch)
    builder.add_node("girl_profile_fetch", girl_profile_fetch)
    builder.add_node("matchmaking_node", matchmaking_node)

    builder.set_entry_point("intent_detection")

    builder.add_conditional_edges(
        "intent_detection",
        route,
        {
            "general": "general",
            "matchmaking": "profile_site_login",
        },
        )

    #builder.add_edge("profile_site_login", "boy_profile_fetch")
    builder.add_conditional_edges(
        "profile_site_login",
        check_login_success,
        {
            "login_successful": "boy_profile_fetch",
            "login_failed": END,
        },
        )
    
    builder.add_edge("general", END)
    builder.add_conditional_edges(
        "boy_profile_fetch",
        check_boy_profile_fetch_success,
        {
            "successful": "girl_profile_fetch",
            "failed": END,
        },
        )

    # check_girl_profile_fetch_success
    builder.add_conditional_edges(
        "girl_profile_fetch",
        check_girl_profile_fetch_success,
        {
            "successful": "matchmaking_node",
            "failed": END,
        },
        )
    
    builder.add_edge("matchmaking_node", END)

    graph = builder.compile()
    return graph



In [3]:
asyncio.run(agent_main())

Entered Profile Site Login Node
Whole result:meta=None content=[TextContent(type='text', text='Logged in successfully', annotations=None, meta=None)] structuredContent=None isError=False extracted result:[{'type': 'text', 'text': 'Logged in successfully'}]
Profile Site Login successful
Entered boy_profile_fetch Node
Whole result:meta=None content=[TextContent(type='text', text='{\n  "Profile Number": "MB08252101",\n  "Full Name": "Sumit Satish Thakar",\n  "Blood Group": "A+ve",\n  "Height": "5\' 9\\"",\n  "DOB": "12-Apr-1993",\n  "Birth Time": "12:11:00 PM",\n  "Birth Place": "Satara",\n  "Family City": "Pune",\n  "Family State": "Maharashtra",\n  "Family Country": "India",\n  "Interest": "NA",\n  "Education": "P.G Data Science, M.E (Machine Design), B.E (Mechanical Engineering)",\n  "Working Field": "MNC",\n  "Company": "Citi Bank (CSIPL)",\n  "Currency": "Rupees",\n  "Annual Income": "24,00,048 Per Annum",\n  "Work City": "Pune, Maharashtra, India",\n  "Family Income": "20 Lac to 50 

Exciting! As a matrimonial assistant, my goal is to help you find your perfect match. I'd be happy to analyze the two profiles and provide some insights.

To get started, could you please share the two profiles with me? These can include information such as:

* Career goals (e.g., industry, job title, aspirations)
* Values (e.g., family-oriented, adventurous, spiritual)
* Personality traits (e.g., optimistic, analytical, creative)
* Education background
* Interests and hobbies

The more information you provide, the better I can match these two profiles based on their values and career goals.

Once I have this information, I'll use my expertise to analyze the compatibility of these two individuals. This may involve identifying commonalities in:

* Shared values and priorities
* Complementary strengths and weaknesses
* Career paths that align with each other's goals

I'll then provide a detailed report highlighting the potential match between these two profiles. This will help you unders

In [ ]:
import streamlit as st

async def agent_main_with_UI():
    #--------------------------
    # Basic Initialisation
    #---------------------------
    mcp_client = MCP_ChatBot(15)
    session_state = AgentState()
    session_state["mcp_client"] = mcp_client
    session_state["stage"] = "input"
    
    # -----------------------------
    # Page Config + Styling
    # -----------------------------
    st.set_page_config(page_title="Matrimonial AI", layout="wide")

    st.markdown("""
        <style>
        body {
            background-color: #f8f5f2;
        }
        .main-title {
            font-size: 36px;
            font-weight: 700;
            color: #7b2cbf;
        }
        .sub-title {
            font-size: 18px;
            color: #555;
        }
        .card {
            background-color: #ffffff;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0px 2px 10px rgba(0,0,0,0.05);
        }
        </style>
    """, unsafe_allow_html=True)

    # -----------------------------
    # Session State Init
    # -----------------------------
    if "stage" not in st.session_state:
        st.session_state.stage = "input"  # input → result → chat

    if "messages" not in st.session_state:
        st.session_state.messages = []

    if "analysis_result" not in st.session_state:
        st.session_state.analysis_result = None

    # -----------------------------
    # Sidebar
    # -----------------------------
    with st.sidebar:
        st.title("⚙️ Settings")

        model_choice = st.selectbox(
            "Choose AI Model",
            ["LLaMA 3 (Ollama)", "OpenAI"]
        )

        if st.button("🔄 Reset Context"):
            st.session_state.stage = "input"
            st.session_state.messages = []
            st.session_state.analysis_result = None
            st.rerun()

    # -----------------------------
    # Header
    # -----------------------------
    st.markdown('<div class="main-title">💍 Matrimonial Match AI</div>', unsafe_allow_html=True)
    st.markdown('<div class="sub-title">Smart compatibility insights powered by AI</div>', unsafe_allow_html=True)
    st.write("")

    # -----------------------------
    # Stage 1: URL Input
    # -----------------------------
    if st.session_state.stage == "input":
        st.markdown("### 🔗 Enter Profile URL")

        profile_url = st.text_input("Profile URL", placeholder="https://example.com/profile")

        if st.button("Analyze Profile"):
            if profile_url:
                # Simulate LangGraph call
                with st.spinner("Analyzing profile..."):
                    # Replace this with real LangGraph invocation
                    result = {
                        "score": 78,
                        "summary": "Good compatibility based on education, career alignment, and values.",
                        "red_flags": ["Different long-term location preference"]
                    }

                    st.session_state.analysis_result = result
                    st.session_state.stage = "result"
                    st.rerun()
            else:
                st.warning("Please enter a valid URL")

    # -----------------------------
    # Stage 2: Result Display
    # -----------------------------
    elif st.session_state.stage == "result":
        result = st.session_state.analysis_result

        st.markdown("### 📊 Match Analysis")

        st.markdown(f"""
        <div class="card">
            <h3>Compatibility Score: {result['score']}%</h3>
            <p>{result['summary']}</p>
            <b>⚠️ Red Flags:</b>
            <ul>
                {''.join([f"<li>{rf}</li>" for rf in result['red_flags']])}
            </ul>
        </div>
        """, unsafe_allow_html=True)

        st.write("")

        if st.button("💬 Continue with Chat"):
            st.session_state.stage = "chat"
            st.rerun()

    # -----------------------------
    # Stage 3: Chat Interface
    # -----------------------------
    elif st.session_state.stage == "chat":
        st.markdown("### 💬 Ask Questions About This Profile")

        # Display chat history
        for msg in st.session_state.messages:
            with st.chat_message(msg["role"]):
                st.write(msg["content"])

        user_input = st.chat_input("Ask something about the match...")

        if user_input:
            # Add user message
            st.session_state.messages.append({"role": "user", "content": user_input})

            with st.chat_message("user"):
                st.write(user_input)

            # Simulate LLM response (replace with LangGraph call)
            with st.chat_message("assistant"):
                with st.spinner("Thinking..."):
                    response = f"(Model: {model_choice})\n\nThis is a sample response analyzing: '{user_input}'"

                    st.write(response)

            st.session_state.messages.append({"role": "assistant", "content": response})